# 3. Two backend models behind one alias

Configure a model alias with two compatible Chat Completions targets at equal priority and weight; see **Configure a two-model pool** in the repository README. Use the same three settings as the other notebooks in the root `.env`: `AI_GATEWAY_CHAT_URL`, `AI_GATEWAY_API_KEY`, and `AI_GATEWAY_MODEL_ALIAS`. Set the shared model alias to your pooled alias; no separate pool setting is needed. Shell environment variables take precedence. This notebook always calls that **one pooled alias**. It sends at most six sequential requests with a long repeated prompt; a 1:1 weight is a *long-run preference*, not a guarantee of three answers from each backend. The longer prompts use more tokens and can incur additional charges, so use a non-production alias and check your quota and token limits.

**Expected result:** The output lists each response's reported model ID, prompt tokens, and cached tokens when available, and summarizes the model counts. Compare these observations with the alias targets and per-backend telemetry; wording and reported IDs alone are not definitive routing evidence. Run the cells again to replace output from a previous version of this notebook.

In [ ]:
import os
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import dotenv_values

notebook_path = globals().get('__vsc_ipynb_file__')
search_roots = [Path.cwd()]
if notebook_path:
    search_roots.insert(0, Path(notebook_path).parent)
lab_root = next(
    (directory for start in search_roots for directory in (start, *start.parents)
     if (directory / '.env.example').is_file() and (directory / 'notebooks').is_dir()),
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
)
env_path = lab_root / '.env'
file_settings = dotenv_values(env_path, encoding='utf-8-sig') if env_path.exists() else {}
required = ('AI_GATEWAY_CHAT_URL', 'AI_GATEWAY_API_KEY', 'AI_GATEWAY_MODEL_ALIAS')
settings = {name: (os.environ[name] if name in os.environ else (file_settings.get(name) or '')).strip()
            for name in required}

chat_url, api_key, alias = (settings[name] for name in required)
missing = [name for name in required if not settings[name]]
del file_settings, settings
if missing:
    status = 'found' if env_path.is_file() else 'not found'
    blank_overrides = [name for name in missing if name in os.environ]
    hint = (f" Blank kernel environment variables override .env: {', '.join(blank_overrides)}."
            ' Remove them or restart the kernel after changing the launching environment.'
            if blank_overrides else '')
    raise RuntimeError(f"Missing or blank settings: {', '.join(missing)}. "
                       f'Checked {env_path} ({status}). Set the missing values there or in the kernel environment.' + hint)
parsed = urlparse(chat_url)
if parsed.scheme != 'https' or not parsed.netloc or not parsed.path.endswith('/chat/completions'):
    raise ValueError('Use the full HTTPS Chat completions endpoint copied from AI Gateway.')
print('Gateway:', parsed.hostname, '| pooled alias:', alias)


## Compare individual answers and prompt-cache usage

Each request has the **same URL, alias, and long prompt prefix**. [Azure OpenAI prompt caching](https://learn.microsoft.com/azure/foundry/openai/how-to/prompt-caching) is enabled by default for supported models and requires at least 1,024 input tokens with the first 1,024 tokens identical. Check the reported `prompt_tokens` to verify the request length; a later `cached_tokens` value greater than zero indicates a cache hit. Zero indicates no reported hit for that request, while `not reported` means the response did not include the field. The first request may warm the cache, but a hit is not guaranteed. Pool routing can send requests to different backends, which may each need to warm their cache. To isolate caching, try a separate non-production alias targeting one supported model, then rerun the setup cell.

The `model` field, if returned, is provider-reported; it may identify the actual backend, contain the alias, or be absent. Differences in wording *do not on their own establish* that requests reached different backends.

In [ ]:
stable_prefix = (
    'An AI gateway routes requests to models and applies governance policies. '
    'It can provide authentication, observability, and traffic management. '
) * 200
observations = []
for number in range(1, 7):
    response = requests.post(
        chat_url,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json={'model': alias, 'messages': [
            {'role': 'user', 'content': stable_prefix + '\nGive one short, original analogy for an AI gateway.'}
        ]},
        timeout=90,
    )
    if response.status_code == 429:
        print('Stopped at request', number, 'after a 429; check the gateway policy and backend quota.')
        break
    response.raise_for_status()
    result = response.json()
    answer = result['choices'][0]['message']['content']
    usage = result.get('usage') or {}
    prompt_tokens = usage.get('prompt_tokens')
    cached_tokens = (usage.get('prompt_tokens_details') or {}).get('cached_tokens')
    observations.append({'request': number, 'reported_model': result.get('model'), 'answer': answer,
                         'prompt_tokens': prompt_tokens, 'cached_tokens': cached_tokens})
    print(f"{number:02d} | model={result.get('model') or 'not reported'}"
          f" | prompt_tokens={prompt_tokens if prompt_tokens is not None else 'not reported'}"
          f" | cached_tokens={cached_tokens if cached_tokens is not None else 'not reported'}"
          f" | {answer[:180]}")


## Interpret the evidence

The counts below refer to **reported response model IDs**, not proven backend resource IDs. Two distinguishable IDs are consistent with requests reaching different models, but backend telemetry is needed to verify selection. If one ID appears, it does not disprove pooling: both deployments might report the same model, or a small sample might select only one target. Check AI Gateway telemetry grouped by backend model after enabling a telemetry destination. Avoid inferring routing from wording differences alone.

In [ ]:
counts = Counter(item['reported_model'] or '<not reported>' for item in observations)
print('Responses by reported model:', dict(counts))
print('Completed requests:', len(observations), '(same alias:', alias + ')')
if len(counts) < 2:
    print('Confirm both backend resource IDs, weights, and actual routing in gateway telemetry.')
